# arc3: ARC-AGI-3 agent

Generated by `src/arc3/notebook.py`; edit the source and rebuild, never this notebook.

In [ ]:
# 0. Mode, budget, configuration
import os, sys, time, threading, traceback
NOTEBOOK_START = time.time()
HARD_WALL = NOTEBOOK_START + 9 * 3600     # Kaggle limit; the runner ends games 15 min earlier
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
OFFLINE_SUBSET = ['ls20', 'ft09', 'tu93', 'vc33']        # offline only: None=all, int=first n, or list of id prefixes
PER_GAME_CAP_SECONDS = 1800.0
MAX_ACTIVE = 10
PORT = 30000
WORKING = "/kaggle/working"
ARC3_ROOT = WORKING                        # package unpacked to /kaggle/working/arc3
LOG_DIR = f"{WORKING}/runs/kaggle"
SERVE_LOG = f"{WORKING}/serve.log"
COMPETITION_DIR = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
os.environ["ONLY_RESET_LEVELS"] = "true"   # RESET keeps the level (must be set before arc_agi import)
os.environ["MPLBACKEND"] = "Agg"
os.environ.setdefault("RECORDINGS_DIR", f"{WORKING}/recordings")
print(f"TRUE_SUBMISSION={TRUE_SUBMISSION} python={sys.version.split()[0]} build=b3d027668446")
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"arc3 needs Python 3.12 (cp312 wheels); kernel has {sys.version}. "
                       "Check docker_image in kernel-metadata.json.")


In [ ]:
# 1. Unpack our source tree (embedded base64 tar.gz of src/arc3)
import base64, hashlib, io, shutil, sys, tarfile
from pathlib import Path
ARC3_TAR_B64 = "..."
ARC3_TAR_SHA256 = "b3d027668446342134d5b5d4c4a898a2a5105cb383941a712b625036d0accb34"
_data = base64.b64decode(ARC3_TAR_B64)
if hashlib.sha256(_data).hexdigest() != ARC3_TAR_SHA256:
    raise RuntimeError("embedded arc3 source archive is corrupt (sha256 mismatch)")
shutil.rmtree(Path(ARC3_ROOT) / "arc3", ignore_errors=True)
with tarfile.open(fileobj=io.BytesIO(_data), mode="r:gz") as _tar:
    _tar.extractall(ARC3_ROOT, filter="data")
if ARC3_ROOT not in sys.path:
    sys.path.insert(0, ARC3_ROOT)
print("arc3 source:", sorted(p.name for p in (Path(ARC3_ROOT) / "arc3").iterdir()))


In [ ]:
# 2. Install arc-agi offline from the competition wheelhouse
import subprocess
_wheels = f"{COMPETITION_DIR}/arc_agi_3_wheels"
if not Path(_wheels).is_dir():
    raise RuntimeError(f"competition wheelhouse missing at {_wheels}; is the competition attached?")
_r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
                     "--no-warn-conflicts", "--disable-pip-version-check",
                     "--find-links", _wheels, "arc-agi"], capture_output=True, text=True)
if _r.returncode:
    raise RuntimeError("offline pip install of arc-agi failed:\n" + _r.stdout[-3000:] + _r.stderr[-3000:])
import arc_agi, arcengine, numpy
print("arc_agi", getattr(arc_agi, "__version__", "?"), "numpy", numpy.__version__)


In [ ]:
# 3. Warm the page cache for the model shards in the background (wheelhouse: in the server thread)
from arc3 import kaggle_serve as ks
PREFETCH_THREAD = ks.prefetch_in_background(ks.MODEL_DIR, ks.DRAFT_MODEL_DIR, delay=60, threads=3)


In [ ]:
# 4. Launch SGLang in the background (install + start); games start right away (keeper RESETs)
SERVER = {"proc": None, "error": None}
SERVER_DEAD = threading.Event()

def _launch():
    try:
        ks.prefetch(ks.WHEELHOUSE_DIR, threads=16)    # the offline install reads all of it
        SERVER["proc"] = ks.launch_sglang(SERVE_LOG, port=PORT)
        ready_at = None
        while True:              # watchdog: a dead server ends the run instead of idling for hours
            if SERVER["proc"].poll() is not None:
                raise RuntimeError(f"SGLang exited with code {SERVER['proc'].returncode}")
            if ready_at is None and ks.healthy(PORT):
                ready_at = time.time()
                ks.say(f"SGLang READY {ready_at - NOTEBOOK_START:.0f}s after notebook start")
            time.sleep(10)
    except Exception:
        SERVER["error"] = traceback.format_exc()
        ks.say("!!! SGLANG FAILED !!!\n" + SERVER["error"] + "\n--- serve.log tail ---\n"
               + ks.log_tail(SERVE_LOG))
        SERVER_DEAD.set()

SERVER_THREAD = threading.Thread(target=_launch, name="sglang", daemon=True)
SERVER_THREAD.start()


In [ ]:
# 5. System monitor (RAM / VRAM / disk + scheduler state every 10 minutes)
from arc3.scheduler import PriorityGate
GATE = PriorityGate(max_active=MAX_ACTIVE)
MONITOR = ks.start_monitor(600, extra=lambda: {"healthy": ks.healthy(PORT), **GATE.snapshot()})


In [ ]:
# 6. Play: competition gateway (rerun) or bundled environment files (offline)
from arc3 import kaggle_main
RUN_ERROR, RUN_SUMMARY = None, None
try:
    RUN_SUMMARY = kaggle_main.main(
        competition=TRUE_SUBMISSION, run_end_epoch=HARD_WALL, log_dir=LOG_DIR,
        llm_url=f"http://127.0.0.1:{PORT}/v1", model_name=ks.SERVED_MODEL_NAME,
        model_ready=lambda: ks.healthy(PORT), offline_subset=OFFLINE_SUBSET,
        per_game_cap_seconds=PER_GAME_CAP_SECONDS, gate=GATE, abort=SERVER_DEAD)
except BaseException:
    RUN_ERROR = traceback.format_exc()
    print("!!! RUN FAILED !!!\n" + RUN_ERROR, flush=True)


In [ ]:
# 7. Offline runs still need a submission.parquet; in a rerun the gateway produces the real one
if not TRUE_SUBMISSION:
    kaggle_main.write_dummy_submission(f"{WORKING}/submission.parquet")


In [ ]:
# 8. Always stop the server, then surface failures
ks.stop_server(SERVER.get("proc"))
print(ks.system_line())
if SERVER.get("error") or RUN_ERROR:
    print("--- serve.log tail ---\n" + ks.log_tail(SERVE_LOG, 20000))
if RUN_SUMMARY is not None:
    print({k: RUN_SUMMARY[k] for k in ("games", "ok", "errors", "hung", "levels_completed", "aborted")})
problem = RUN_ERROR or SERVER.get("error")
if problem and not TRUE_SUBMISSION:
    # In a rerun we never raise here: whatever was played is already on the gateway scorecard.
    raise RuntimeError("arc3 run failed; see the log above")
